In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
import analysis as an

import numpy as np
import matplotlib.pyplot as plt

# nDGP signal in the HOD galaxy power spectrum and marked power spectrum

$S_X(k) = \langle X_\mathrm{nDGP}\rangle(k) / \langle X_\mathrm{GR}\rangle(k) - 1$ for the redshift-space
multipoles $X = P_0, P_2$ of the HOD mocks made by `jobs/hod_mock.sh` (measured as in the EMC/acm
pipeline), averaged over HOD seeds, then the same for the marked power spectrum $M_0, M_2$ of the
same mocks. The fine k-bins of the measurement are rebinned to width `dk`, weighting by the number
of modes.

The curve is the ratio of the seed-averaged multipoles; the band is its error from the
seed-to-seed scatter of the per-seed $S_X$ (standard deviation / $\sqrt{N_\mathrm{seeds}}$, shown
for 2 seeds or more). Both models use the same HOD parameters and seeds; their galaxy number
densities and biases can differ (see the printed densities), which enters $S_X$ together with the
effect of gravity.

In [ ]:
target = 'Abacus_Small'
redshift = 0.5
realization = 's1234'
name = 'LRG_seed{seed}'      # NAME in jobs/hod_mock.sh
seeds = list(range(1, 21))                 # HOD seeds averaged (the same for both models), e.g. list(range(1, 11))
gr_model = 'LCDM'
ndgp_model = 'nDGP_N1.2'
ells = (0, 2)                # X = P_ell
dk = 0.01                    # rebinned k-bin width [h/Mpc]
k_max = 0.5                  # largest k shown [h/Mpc]
show_seeds = True            # also draw S_X of each seed (thin lines)

## Load

In [ ]:
import json


def hod_metadata(model, seed, statistic):
    # metadata of one HOD measurement ('PS' or 'MPS'): run folder on scratch, or the kept copy
    args = (target, model, realization, redshift, name.format(seed=seed), statistic)
    for spectrum_file in (hp.hod_spectrum_file(*args), hp.hicola_hod_spectrum_file(*args)):
        meta_file = Path(str(spectrum_file) + '.json')
        if meta_file.exists():
            return json.loads(meta_file.read_text())
    return {}


def load_signal(statistic):
    # S_X of the seed-averaged multipoles, its error and the per-seed S_X, for 'PS' or 'MPS'
    P = {}
    for model in (gr_model, ndgp_model):
        k_model, P[model] = an.load_hod_spectra(target, model, realization, redshift, name, seeds, dk, statistic)
        densities = [hod_metadata(model, seed, statistic).get('number_density_h3_Mpc3', np.nan) for seed in seeds]
        print(f'{model}: {len(seeds)} seeds, galaxy density {np.mean(densities):.3e} (h/Mpc)^3 '
              f'(min {np.min(densities):.3e}, max {np.max(densities):.3e})')
        if model == gr_model:
            k = k_model
        elif not (len(k_model) == len(k) and np.allclose(k_model, k)):
            raise ValueError('the two models have different k-bins')

    keep = k <= k_max
    S, S_err, S_seeds = {}, {}, {}
    for ell in ells:
        P_gr, P_ndgp = P[gr_model][ell][:, keep], P[ndgp_model][ell][:, keep]
        S[ell] = P_ndgp.mean(axis=0) / P_gr.mean(axis=0) - 1
        S_seeds[ell] = P_ndgp / P_gr - 1   # per seed, same seed in both models
        S_err[ell] = S_seeds[ell].std(axis=0, ddof=1) / np.sqrt(len(seeds)) if len(seeds) > 1 else None
    return k[keep], S, S_err, S_seeds


k, S, S_err, S_seeds = load_signal('PS')

## $S_X(k)$

In [ ]:
def plot_signal(k, S, S_err, S_seeds, letter):
    # letter: 'P' for the power spectrum, 'M' for the marked power spectrum
    n_seeds = len(seeds)
    fig, axes = plt.subplots(1, len(ells), figsize=(3.4 * len(ells), 3.0), dpi=200, constrained_layout=True)
    for ax, ell in zip(np.atleast_1d(axes), ells):
        if show_seeds and n_seeds > 1:
            for S_seed in S_seeds[ell]:
                ax.plot(k, S_seed, color='0.7', lw=0.4, alpha=0.6)
        if S_err[ell] is not None:
            ax.fill_between(k, S[ell] - S_err[ell], S[ell] + S_err[ell], color='#c47f2b', alpha=0.25, lw=0,
                            label=rf'error of the mean ({n_seeds} seeds)')
        ax.plot(k, S[ell], color='#c47f2b', lw=1.4, label='seed-averaged' if n_seeds > 1 else f'seed {seeds[0]}')
        ax.axhline(0, color='0.4', lw=0.8, ls=':')
        ax.set_xscale('log')
        ax.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$')
        ax.set_ylabel(rf'$S_{{{letter}_{ell}}}(k) = {letter}_{ell}^\mathrm{{nDGP}} / {letter}_{ell}^\mathrm{{GR}} - 1$')
        ax.set_title(rf'${letter}_{ell}$, {target}, z = {redshift}', fontsize=9)
        ax.legend(fontsize=7, frameon=False)
        ax.minorticks_on()
        ax.grid(True, which='major', ls=':', color='0.82', lw=0.45)
        ax.tick_params(which='both', direction='in', top=True, right=True)
    plt.show()


plot_signal(k, S, S_err, S_seeds, 'P')

## Marked power spectrum

$M_\ell$: multipoles of the marked power spectrum of the same mocks (`<name>_MPS.txt`), with the mark
$m = \sum_n c_n \delta_R^n$ of the galaxy density contrast smoothed on the scale $R$, normalised by its
mean over the galaxies (acm's estimator). The mark settings of the files are printed and must be
the same for all seeds and both models.

In [ ]:
marks = {(model, seed): hod_metadata(model, seed, 'MPS').get('mark', {}) for model in (gr_model, ndgp_model) for seed in seeds}
settings = {(mark.get('rcut'), tuple(mark.get('coefficients', ()))) for mark in marks.values()}
if len(settings) != 1:
    raise ValueError(f'the marked spectra use different marks (R, coefficients): {settings}')
(rcut, coefficients), = settings
print(f'Mark: R = {rcut} Mpc/h, coefficients {coefficients}')
for model in (gr_model, ndgp_model):
    m_bar = [marks[model, seed].get('m_bar', np.nan) for seed in seeds]
    print(f'{model}: m_bar {np.mean(m_bar):.4f} (min {np.min(m_bar):.4f}, max {np.max(m_bar):.4f})')

k_M, S_M, S_M_err, S_M_seeds = load_signal('MPS')

In [ ]:
plot_signal(k_M, S_M, S_M_err, S_M_seeds, 'M')